# Env39 — Planar Geometry, Circuity and Orientation Metrics

## 1. Purpose

This notebook presents the Phase 7G planar geometry metrics for Env39: physical-edge length, endpoint chord, circuity, chord and LineString-segment orientation, Shannon orientation entropy, and fourfold orientation order. The tested package performs every scientific calculation so this notebook remains an explanatory research interface.

In [ ]:
ENVIRONMENT = "env39"
CANONICAL_RUN = "latest"

## 2. Why this phase uses planar geometry

GeoGami coordinates form a local Cartesian engineering system, not longitude and latitude. Euclidean geometry is therefore calculated directly in that plane. Geographic great-circle lengths, geographic bearings, nearest-node routines, and invented WGS84 coordinates would be scientifically invalid here.

## 3. Selected canonical run/provenance

The existing Phase 7 resolver checks the canonical file SHA-256, scientific-content signature, topology signature, validation status, and manifest paths before geometry analysis.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
PROJECT_ROOT = next((candidate for candidate in (start, *start.parents)
                     if (candidate / "environment.yml").is_file()
                     and (candidate / "src" / "geogami_morphology").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Repository root not found. Start Jupyter from the repository or notebooks directory.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from IPython.display import HTML, display
from geogami_morphology.metrics_geometry import analyze_geometry, load_geometry_config

result = analyze_geometry(ENVIRONMENT, canonical_run=CANONICAL_RUN, project_root=PROJECT_ROOT, publish=False)
config = load_geometry_config(project_root=PROJECT_ROOT)
metric = result.calculation.metric
print("REPOSITORY_DISCOVERY: PASS")
print("Canonical run:", result.selection.run_id)
print("Canonical file SHA-256:", result.selection.file_sha256)
print("Scientific-content signature:", result.selection.scientific_content_signature)
print("Topology signature:", result.selection.topology_signature)
print("CANONICAL_GEOMETRY_RESOLUTION: PASS")

## 4. Coordinate system and units

All distances are **local units**, not metres. Orientations use a planar north-clockwise convention: 0° is North and 90° is East. Streets are axes rather than directed movements, so angles are normalized to `[0°,180°)` and reversing an edge does not change its orientation.

In [ ]:
print("Coordinate system: GeoGami Local Cartesian")
print("Distance units: local units (NOT metres)")
print(f"Axial bins: {config.orientation_bins} x {config.orientation_bin_width_degrees} degrees")
print("Configuration SHA-256:", config.file_sha256)
print("PLANAR_CONFIGURATION: PASS")

## 5. Edge geometry length

For each physical canonical edge, $L_i$ is recomputed as the actual planar `LineString.length`. The stored derived length is checked within the configured floating tolerance rather than trusted blindly.

In [ ]:
edge_frame = pd.DataFrame(result.calculation.edge_rows)
length_names = ["total_network_length_local", "mean_edge_length_local", "median_edge_length_local", "std_edge_length_local", "minimum_edge_length_local", "maximum_edge_length_local", "q1_edge_length_local", "q3_edge_length_local"]
display(pd.DataFrame([{"metric": name, "value": metric(name), "units": "local units"} for name in length_names]))
print("EDGE_LENGTH_METRICS: PASS")

## 6. Chord length

The endpoint chord is $D_i = \sqrt{(x_v-x_u)^2 + (y_v-y_u)^2}$ using canonical `u` and `v` node Points. Internal LineString vertices are not graph nodes. Edges with $D_i$ at or below the configured tolerance are explicitly classified as zero-chord.

In [ ]:
chord_names = ["total_chord_length_local", "mean_chord_length_local", "median_chord_length_local", "minimum_chord_length_local", "maximum_chord_length_local", "zero_chord_edge_count"]
display(pd.DataFrame([{"metric": name, "value": metric(name)} for name in chord_names]))
print("CHORD_LENGTH_METRICS: PASS")

## 7. Circuity definitions

For every positive-chord edge, **edge circuity** is $c_i=L_i/D_i$. A straight edge has approximately 1; larger values indicate geometric indirectness. **Mean edge circuity** is $\mathrm{mean}(L_i/D_i)$, whereas **network circuity** is $\sum L_i / \sum D_i$. They are different statistics. The unsupported historical H4 aggregate is not reused.

In [ ]:
circuity_names = ["mean_edge_circuity", "median_edge_circuity", "std_edge_circuity", "minimum_edge_circuity", "maximum_edge_circuity", "q1_edge_circuity", "q3_edge_circuity", "network_circuity"]
display(pd.DataFrame([{"metric": name, "value": metric(name)} for name in circuity_names]))
assert all(value >= 1 - config.circuity_floating_tolerance for value in edge_frame.circuity.dropna())
print("PLANAR_CIRCUITY: PASS")

## 8. Edge-chord orientation

Chord orientation describes the overall endpoint-to-endpoint street axis. It uses one equal-weight observation per positive-chord physical edge. This summary deliberately cannot describe the local orientations inside a curved or bent street.

In [ ]:
display(edge_frame[["edge_id", "chord_length_local", "chord_orientation_deg"]].head(12))
assert edge_frame.chord_orientation_deg.dropna().between(0, 180, inclusive="left").all()
print("CHORD_ORIENTATION: PASS")

## 9. LineString-segment orientation

Every positive consecutive coordinate pair contributes a local axial orientation and its planar segment length. Segment weights therefore let a bent physical edge contribute multiple local directions without treating geometry vertices as graph nodes.

In [ ]:
segment_frame = pd.DataFrame(result.calculation.segment_rows)
display(segment_frame.head(12))
assert abs(segment_frame.segment_length_local.sum() - metric("total_network_length_local")) < config.length_comparison_tolerance_local
print("SEGMENT_ORIENTATION: PASS")

## 10. Orientation histogram and binning

The 180° axial domain uses 36 half-open 5° bins. Angles are normalized modulo 180; values within the configured 1e-9° floating tolerance of an exact boundary are snapped to it, then assigned by `floor(angle / 5°)`. The chord histogram uses equal-edge probabilities; the segment histogram uses segment-length probabilities.

In [ ]:
chord_hist = pd.DataFrame(result.calculation.chord_histogram)
segment_hist = pd.DataFrame(result.calculation.segment_histogram)
display(chord_hist.loc[chord_hist.probability > 0])
display(segment_hist.loc[segment_hist.probability > 0])
assert abs(chord_hist.probability.sum() - 1) < 1e-12
assert abs(segment_hist.probability.sum() - 1) < 1e-12
print("AXIAL_HISTOGRAMS: PASS")

## 11. Shannon orientation entropy

For non-zero bin probabilities, $H=-\sum_i p_i\ln(p_i)$ using the natural logarithm. With 36 bins, $H_{max}=\ln(36)$ and normalized entropy is $H/H_{max}$. Lower values mean orientation concentration; higher values mean diversity. Entropy alone is not curvature.

In [ ]:
entropy_names = ["chord_orientation_entropy", "chord_orientation_normalized_entropy", "segment_length_weighted_orientation_entropy", "segment_length_weighted_normalized_entropy"]
display(pd.DataFrame([{"metric": name, "value": metric(name)} for name in entropy_names]))
print("SHANNON_ENTROPY: PASS")

## 12. Fourfold orientation order phi

The orthogonal order parameter is $\phi=|\sum_i w_i\exp(i4\theta_i)/\sum_i w_i|$, with $\theta$ in radians. Directions 90° apart reinforce. Values near 1 show strong fourfold order; values near 0 show weak fourfold order. Chord φ uses equal streets, while segment φ uses local segment length.

In [ ]:
display(pd.DataFrame([
    {"metric": "phi_chord", "value": metric("phi_chord"), "weighting": "equal physical edge"},
    {"metric": "phi_segment_length_weighted", "value": metric("phi_segment_length_weighted"), "weighting": "segment length"},
]))
print("FOURFOLD_ORIENTATION_ORDER: PASS")

## 13. Geometry metric summary

Long-form result rows retain units, formula/method, weighting, population, and interpretation. This prevents chord and segment results or mean and aggregate circuity from being silently conflated.

In [ ]:
display(pd.DataFrame(result.calculation.geometry_summary_rows))
display(pd.DataFrame(result.calculation.orientation_summary_rows))

## 14. Scientific figures

The exported SVGs are deterministic and headless-safe. The rose diagram explicitly mirrors axial `[0°,180°)` observations as undirected axes; it is not a 360° traffic-bearing plot.

In [ ]:
figure_names = ["edge_length_distribution.svg", "edge_circuity_distribution.svg", "chord_orientation_histogram.svg", "chord_orientation_rose.svg", "segment_length_weighted_orientation_histogram.svg"]
for name in figure_names:
    figure_path = result.output_dir / "figures" / name
    if not figure_path.is_file():
        raise RuntimeError(f"Missing published figure: {name}")
    display(HTML(figure_path.read_text(encoding="utf-8")))
print("HEADLESS_GEOMETRY_FIGURES: PASS")

## 15. Exported artifacts

Stable CSV, JSON, methodology, figure, and manifest artifacts are published under the selected canonical run's `geometry/` directory. Phase 7D and Phase 7F files are hash-checked and left untouched.

In [ ]:
required = ["geometry_metrics_summary.csv", "geometry_metrics_summary.json", "edge_geometry_metrics.csv", "segment_orientation_metrics.csv", "orientation_metrics_summary.csv", "orientation_metrics_summary.json", "geometry_methodology.json", "geometry_analysis_manifest.json"]
missing = [name for name in required if not (result.output_dir / name).is_file()]
if missing:
    raise RuntimeError(f"Missing published geometry artifacts: {missing}. Run scripts/run_geometry_metrics.py first.")
print("Artifacts directory:", result.output_dir.relative_to(PROJECT_ROOT))
print("EXPORTED_GEOMETRY_ARTIFACTS: PASS")

## 16. Interpretation for grid-like Env39

Env39 has highly concentrated orthogonal local segment orientations. Chord results are slightly less ordered because an endpoint chord summarizes a bent edge as one diagonal-like axis. These are structural descriptions, not claims that Env39 is objectively better or more navigable.

## 17. How these metrics will later compare to Env38

The same formulas, tolerances, bins, and weights can later quantify whether a topology-controlled curvilinear Env38 has different excess length, circuity, orientation diversity, and fourfold order. Comparative conclusions must wait until Env38 is analyzed through the same canonical workflow.

## 18. Reproducibility summary

Canonical identities, the configuration SHA-256, exact formulas, orientation convention, binning, weighting, software versions, Phase 7F preservation hashes, and output hashes are recorded in `geometry_analysis_manifest.json`.

In [ ]:
assert (metric("node_count"), metric("physical_edge_geometry_count")) == (46, 69)
assert result.selection.topology_signature == "4AF93910532EB430103267DAECF73B3ADC3B3D1599471EDF6B1632B6C5F898CB"
assert 0 <= metric("phi_chord") <= 1 and 0 <= metric("phi_segment_length_weighted") <= 1
print("Geographic OSMnx functions: NOT USED")
print("Phase 7H: NOT STARTED")
print("NOTEBOOK_03_EXECUTION: PASS")